# Data Scale and Variance Profiler

This notebook evaluates the underlying scale, mean, and variance of any target dataset. 
We use this to contextualise our Mean Absolute Error (MAE) validation scores. 

In [1]:
import sys
from pathlib import Path
import pandas as pd

# 1. Dynamically locate the project root (assuming this notebook is in Data_Pipeline)
current_dir = Path.cwd()
project_root = current_dir.parent
sys.path.append(str(project_root))

# 2. Import paths
from Config.Paths import HACK_NOI_MONTHLY_V2

# 3. Define the target file (make this general purpose so it can be swapped later)
TARGET_CSV_PATH = HACK_NOI_MONTHLY_V2

print(f"Target file configured: {TARGET_CSV_PATH.name}")

Target file configured: NoI_monthly_v2.csv


## Load and Standardise Data
We load the general-purpose CSV, isolate the numerical columns, and standardise the date parsing to handle variations (e.g., `MM/YYYY` vs `MMM-YY`).

In [2]:
df = pd.read_csv(TARGET_CSV_PATH)

# Identify the date column (assumed to be index 0) and numerical columns
date_col = df.columns[0]
numeric_cols = [col for col in df.columns if col != date_col]

# Parse dates robustly
df['parsed_date'] = pd.to_datetime(df[date_col], format='%m/%Y', errors='coerce')
if df['parsed_date'].isna().all():
    df['parsed_date'] = pd.to_datetime(df[date_col], format='%b-%y', errors='coerce')

print(f"Dataset loaded: {len(df)} rows, {len(numeric_cols)} numerical features.")
print(f"Date range: {df['parsed_date'].min().date()} to {df['parsed_date'].max().date()}")

Dataset loaded: 174 rows, 988 numerical features.
Date range: 2011-07-01 to 2025-12-01


## Global Descriptive Statistics
Calculates the historical baseline for every numerical feature. We isolate the mean, standard deviation, and maximum values to understand the typical monthly volume of each category.

In [3]:
# Generate standard descriptive statistics
global_stats = df[numeric_cols].describe().T

# Filter down to the most relevant metrics for MAE contextualisation
profile_df = global_stats[['mean', 'std', 'max']].copy()
profile_df.columns = ['Historical_Monthly_Mean', 'Standard_Deviation', 'Historical_Max']

# Sort by highest volume categories to see where the bulk of the error likely stems from
profile_df = profile_df.sort_values(by='Historical_Monthly_Mean', ascending=False)

display(profile_df.head(15))

,Historical_Monthly_Mean,Standard_Deviation,Historical_Max
Malware-ALL,742.293103,739.907814,2527.0
Account Hijacking-ALL,142.459770,157.675771,646.0
Vulnerability-ALL,139.931034,169.530620,891.0
Phishing-ALL,136.465517,155.701012,564.0
Targeted Attack-ALL,131.712644,116.704301,538.0
Ransomware-ALL,125.574713,153.341632,580.0
Others-ALL,101.528736,123.431415,650.0
Data Breach-ALL,79.816092,76.610195,380.0
Trojan-ALL,69.712644,90.700410,447.0
Botnet-ALL,59.235632,73.599186,333.0


## The 2024 Specific Context
To specifically audit the 2024 validation MAE, we filter the dataset to the 2024 calendar year and check the incident volumes. This tells us if 2024 was an unusually quiet or loud year compared to the historical baseline.

In [4]:
# Filter for 2024 specifically
start_2024 = pd.to_datetime("2024-01-01")
end_2024 = pd.to_datetime("2024-12-31")

df_2024 = df[(df['parsed_date'] >= start_2024) & (df['parsed_date'] <= end_2024)]

if not df_2024.empty:
    stats_2024 = df_2024[numeric_cols].describe().T[['mean', 'max']]
    stats_2024.columns = ['2024_Monthly_Mean', '2024_Max']
    
    # Merge with the global profile to compare side-by-side
    comparison_df = profile_df.join(stats_2024)
    comparison_df['Variance_from_History'] = comparison_df['2024_Monthly_Mean'] - comparison_df['Historical_Monthly_Mean']
    
    display(comparison_df.head(15))
else:
    print("No 2024 data found in this specific CSV.")

,Historical_Monthly_Mean,Standard_Deviation,Historical_Max,2024_Monthly_Mean,2024_Max,Variance_from_History
Malware-ALL,742.293103,739.907814,2527.0,1890.666667,2213.0,1148.373563
Account Hijacking-ALL,142.459770,157.675771,646.0,252.166667,532.0,109.706897
Vulnerability-ALL,139.931034,169.530620,891.0,297.583333,460.0,157.652299
Phishing-ALL,136.465517,155.701012,564.0,330.916667,564.0,194.451149
Targeted Attack-ALL,131.712644,116.704301,538.0,235.666667,538.0,103.954023
Ransomware-ALL,125.574713,153.341632,580.0,291.833333,556.0,166.258621
Others-ALL,101.528736,123.431415,650.0,263.416667,650.0,161.887931
Data Breach-ALL,79.816092,76.610195,380.0,129.250000,251.0,49.433908
Trojan-ALL,69.712644,90.700410,447.0,107.083333,191.0,37.370690
Botnet-ALL,59.235632,73.599186,333.0,97.250000,333.0,38.014368
